In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_DAGSLAM"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("nij root:" , nij_root)
print("Output directory:", output_dir)

nij root: /dcs/23/u2200504/thesis/recidivism-causal/data/processed
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/NIJ/graphs_DAGSLAM


In [3]:
#add dagslam implementation to system path
dagslam_path = project_root/"code"/"dagslam"/"DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data"/"_dagslam"
sys.path.append(str(dagslam_path))

#uses DAGSLAM implementation code authored by Yuanyuan Zhao. 
#https://github.com/yuanyuan-zhao-pku/DAGSLAM/blob/main/DAGSLAM%20Causal%20Bayesian%20Network%20Structure%20Learning%20of%20Mixed%20Type%20Data/_dagslam/DAGSLAM.py
#DAGSLAM is an extension of the NOTEARS algorithm developed by Xun Zheng, et al.
import importlib
import DAGSLAM 
importlib.reload(DAGSLAM)

from DAGSLAM import dagslam

#m_vec gives the total number of categories for each multinomial variable
#helper function to infer loss types and generate m_vec for each column
def infer_type(df):
    loss_type=[]
    m_vec=[]

    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        num_unique = len(unique_vals)

        #infer variable type by combination of dtype and cardinality
        if np.issubdtype(x.dtype, np.number):
            #numeric datatypes
            if num_unique == 2 and set(unique_vals).issubset({0,1}): #binary numeric variable
                loss_type.append("logistic")
                m_vec.append(1) # 1 "category"
            else: #continuous 
                loss_type.append("gauss")
                m_vec.append(1)
        else: #non-numeric
            if num_unique == 2:
                #binary categorical 
                loss_type.append("logistic")
                m_vec.append(1)
            else:
                #multi-class categorical
                loss_type.append("multi-logistic")
                m_vec.append(num_unique)
    print("loss_type:", loss_type)
    print("m_vec:", m_vec)
    print("n_cols:", len(df.columns), "len(loss_type):", len(loss_type))
    
    return loss_type, m_vec

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc


def run_dagslam(df, lambda1=0.03, max_iter=100, w_threshold=0.25): #changed lambda 1 0.03 ->0.1 w_threshold 0.25->0.3
    df_clean = df.dropna().copy()
    df_enc = encode_mixed_df(df_clean)
    
    loss_type, m_vec =infer_type(df_clean)
    assert list(df_clean.columns) == list(df_enc.columns)
    X=df_enc.to_numpy(dtype=float)

    start = time.time() 
    W_est = dagslam(X, loss_type=loss_type, m_vec=m_vec, lambda1=lambda1,max_iter=max_iter, w_threshold=w_threshold)
    end = time.time()
    #print(f"DAGSLAM took {(end - start)/60:.2f} minutes")

    return W_est

#draw graphs based on DAGSLAM weighted adjacency matrix
def draw_graph(W, output_path, node_labels=None,threshold=0):
    n = W.shape[0] # number of nodes
    G = nx.DiGraph() # initialise empty directed graph
    
    #initialise default node labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]

    #add nodes to digraph
    for i, name in enumerate(node_labels):
        G.add_node(i, label=name)

    #add edges for surviving weights (thresholding done during the dagslam phase)
    for i in range(n):
        for j in range(n): # for each possible edge 
            w=W[i,j]
            if abs(w)>threshold:
                G.add_edge(i, j, weight=w)

    plt.figure(figsize=(6,6))
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=True, labels={i: node_labels[i] for i in range(n)},
            node_size=800, font_size=8, arrowsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

In [4]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(W, out_path, node_labels=None, threshold=0.0, engine="dot"):

    W = np.asarray(W)
    if W.ndim == 1:
        W = W.reshape(1, 1)

    n = W.shape[0]

    # Default labels use X0, X1,...
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="svg", engine=engine)

    g.attr(rankdir="LR")
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = W[i, j]
            if abs(w) > threshold:
                # You can optionally add weight as label=str(round(w, 2))
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [9]:
csv_path = output_dir / "NIJ_graph_DAGSLAM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy() #adjacency matrix
G = nx.from_pandas_adjacency(adj_df, create_using=nx.DiGraph)

In [6]:
def show_g(G):
    plt.figure(figsize=(6, 6))
    pos = nx.spring_layout(G, seed=0)
    
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_size=800,
        font_size=8,
        arrowsize=10,
    )
    
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [12]:
import itertools
import random

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        W_subset = run_dagslam(df_subset);
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, W_subset))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

score = incompatibility_score(A, k=5, n_subsets=50, seed=42)
print("Approx. incompatibility score:", score)


loss_type: ['logistic', 'logistic', 'logistic', 'logistic', 'logistic']
m_vec: [1, 1, 1, 1, 1]
n_cols: 5 len(loss_type): 5
iter:0
rho:1.0
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.2785231979447857
loss=2.68223452155695
loss=2.6276025341995703
loss=2.6147387145815264
loss=2.6102120871193644
loss=2.6078934827978664
loss=2.6071583527977893
loss=2.6068169490642834
loss=2.6075095036732776
loss=2.6072914034936483
loss=2.606930620849384
loss=2.6066654632628814
loss=2.606371845843081
loss=2.6063183435783324
loss=2.60632282067213
lo

In [14]:
incompat_score = 3

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))

In [15]:
goodness(incompat_score, 5)

% of edges disagreeing on avg: 15.0


In [9]:
import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        adj= run_dagslam(df_subset)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))
    return(frac*100)
    
csv_path = output_dir/ "NIJ_graph_DAGSLAM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

seeds = [42,7,12]
incompat_scores = []
disagree = []
for seed in seeds:
    score = incompatibility_score(A, k = 5, n_subsets=50, seed=seed)
    incompat_scores.append(score)
    disagree.append(goodness(score, 5))

loss_type: ['logistic', 'logistic', 'logistic', 'logistic', 'logistic']
m_vec: [1, 1, 1, 1, 1]
n_cols: 5 len(loss_type): 5
iter:0
rho:1.0
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.2785231979447857
loss=2.68223452155695
loss=2.6276025341995703
loss=2.6147387145815264
loss=2.6102120871193644
loss=2.6078934827978664
loss=2.6071583527977893
loss=2.6068169490642834
loss=2.6075095036732776
loss=2.6072914034936483
loss=2.606930620849384
loss=2.6066654632628814
loss=2.606371845843081
loss=2.6063183435783324
loss=2.60632282067213
lo

In [10]:
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 0.2217105219775452
standard dev of disagreement percentage: 1.1085526098877256


In [11]:
incompat_scores

[3.0, 3.54, 3.22]

In [12]:
disagree

[15.0, 17.7, 16.1]

In [ ]:
import time
csv_path = nij_root / "NIJ_lean_compact_onehot.csv"
df_full = pd.read_csv(csv_path)
node_labels = list(df_full.columns)
d = len(node_labels)

def bootstrap_edge_stability(df, B=20, seed=0):
    rng = np.random.default_rng(seed)
    edge_counts = np.zeros((d, d), dtype=int)

    for b in range(B):
        #sample B rows with replacement from original learning data
        idx = rng.integers(low=0, high=len(df), size=len(df))
        df_boot = df.iloc[idx, :]

        #run DAGSLAM on each bootstrap sample
        W_est = run_dagslam(df_boot)           # shape (d, d), aligned with columns
        W_bin = (np.asarray(W_est) != 0).astype(int)

        #accumulate edge counts
        edge_counts += W_bin

    #convert to percentage appearances
    edge_freq = edge_counts / B
    return edge_freq
    
edge_freq_dagslam = bootstrap_edge_stability(df_full, B=20, seed=42)

#save result to CSV
edge_freq_df = pd.DataFrame(edge_freq_dagslam, index=node_labels, columns=node_labels)
edge_freq_df.to_csv(output_dir / "NIJ_DAGSLAM_edge_stability.csv")
print("Number of edges with freq >= 0.5:",
      np.sum(edge_freq_dagslam >= 0.5))
print("Number of edges with freq >= 0.8:",
      np.sum(edge_freq_dagslam >= 0.8))


In [ ]:
import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        adj= run_dagslam(df_subset)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))
    return(frac*100)
    
csv_path = output_dir/ "NIJ_graph_DAGSLAM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

sizes = [5,10,15]
incompat_scores = []
disagree = []
for size in sizes:
    score = incompatibility_score(A, k = size, n_subsets=50, seed=42)
    incompat_scores.append(score)
    disagree.append(goodness(score, size))

loss_type: ['logistic', 'logistic', 'logistic', 'logistic', 'logistic']
m_vec: [1, 1, 1, 1, 1]
n_cols: 5 len(loss_type): 5
iter:0
rho:1.0
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.465735902799727
loss=3.2785231979447857
loss=2.68223452155695
loss=2.6276025341995703
loss=2.6147387145815264
loss=2.6102120871193644
loss=2.6078934827978664
loss=2.6071583527977893
loss=2.6068169490642834
loss=2.6075095036732776
loss=2.6072914034936483
loss=2.606930620849384
loss=2.6066654632628814
loss=2.606371845843081
loss=2.6063183435783324
loss=2.60632282067213
lo